# Detección de Malware mediante Análisis Estático de Ejecutables

## Introducción a la Detección Estática de Malware

Los enfoques tradicionales basados en firmas fallan ante malware nuevo, variantes polimórficas y ofuscación.
Este cuaderno implementa detección basada en **análisis estático de características**, clasificando
ejecutables como benignos o maliciosos mediante machine learning.

### Contenidos
- **Extracción de características PE**: metadatos, secciones, entropía, importaciones
- **Balanceo de clases con SMOTE**: para manejar datasets desbalanceados  
- **Evaluación de clasificadores**: Árbol de Decisión, Random Forest, SVM
- **Visualización e interpretabilidad** de decisiones del modelo

---

## Generación de Dataset Sintético PE

Se crea un dataset simulado con características de ejecutables reales.
Cada fila representa un archivo PE con sus características extraídas.

### Características extraídas por PE
| Característica | Rango / Tipo | Explicación |
| :--- | :--- | :--- |
| `entry_point` | Dirección (int) | Punto de entrada del programa |
| `num_sections` | 0-15 | Número de secciones en cabecera PE |
| `entropy_max` | 0-8 | Entropía (alta=empaquetado/cifrado) |
| `num_imports` | 0-200 | Funciones importadas |
| `num_dlls` | 0-50 | DLLs linkeditadas |

In [ ]:
# Listing 4.2: Extracción de características de un archivo PE

import pefile
import math
import os

def calcular_entropia(data: bytes) -> float:
    """Calcula la entropía de Shannon de un bloque de bytes."""
    if not data:
        return 0.0
    freq = [data.count(bytes([b])) / len(data) for b in range(256)]
    return -sum(p * math.log2(p) for p in freq if p > 0)

def extraer_caracteristicas_pe(ruta_archivo: str) -> dict:
    """
    Extrae características relevantes de un ejecutable PE.
    Retorna un diccionario con las características o None si falla.
    """
    try:
        pe = pefile.PE(ruta_archivo)
    except pefile.PEFormatError:
        print(f"[!] No es un PE válido: {ruta_archivo}")
        return None

    caracteristicas = {}

    # Cabecera opcional
    caracteristicas['entry_point'] = pe.OPTIONAL_HEADER.AddressOfEntryPoint
    caracteristicas['image_base'] = pe.OPTIONAL_HEADER.ImageBase
    caracteristicas['size_of_image'] = pe.OPTIONAL_HEADER.SizeOfImage
    caracteristicas['size_code_section'] = pe.OPTIONAL_HEADER.SizeOfCode
    caracteristicas['dll_flag'] = pe.OPTIONAL_HEADER.DllCharacteristics

    # Secciones
    caracteristicas['num_sections'] = len(pe.sections)
    entropias = [calcular_entropia(sec.get_data()) for sec in pe.sections]
    caracteristicas['entropia_max'] = max(entropias) if entropias else 0.0
    caracteristicas['entropia_media'] = (
        sum(entropias) / len(entropias) if entropias else 0.0
    )

    # Importaciones / exportaciones
    if hasattr(pe, 'DIRECTORY_ENTRY_IMPORT'):
        caracteristicas['num_importaciones'] = sum(
            len(entry.imports)
            for entry in pe.DIRECTORY_ENTRY_IMPORT
        )
        caracteristicas['num_dlls_importadas'] = len(pe.DIRECTORY_ENTRY_IMPORT)
    else:
        caracteristicas['num_importaciones'] = 0
        caracteristicas['num_dlls_importadas'] = 0

    if hasattr(pe, 'DIRECTORY_ENTRY_EXPORT'):
        caracteristicas['num_exportaciones'] = len(
            pe.DIRECTORY_ENTRY_EXPORT.symbols
        )
    else:
        caracteristicas['num_exportaciones'] = 0

    # Tamaño del archivo
    caracteristicas['file_size'] = os.path.getsize(ruta_archivo)

    pe.close()
    return caracteristicas

print("[OK] Funciones de extracción de características PE definidas.")
print("     - calcular_entropia(data)")
print("     - extraer_caracteristicas_pe(ruta_archivo)")

---

### Compilación del Corpus de Malware

Se sintetiza un dataset de características PE de 800 ejecutables benignos
y 200 maliciosos, balanceando el dataset para evitar sesgos.


In [ ]:
# Listing 4.3: Generar dataset con etiquetas benign/malicious

import os
import pandas as pd

def construir_dataset(directorios: dict) -> pd.DataFrame:
    """
    directorios: {'benign': '/ruta/benign', 'malicious': '/ruta/malicious'}
    Retorna un DataFrame con características y etiqueta.
    """
    registros = []
    for etiqueta, directorio in directorios.items():
        valor_label = 0 if etiqueta == 'benign' else 1
        if not os.path.exists(directorio):
            print(f"[ADVERTENCIA] Directorio no encontrado: {directorio}")
            continue
        for archivo in os.listdir(directorio):
            ruta_completa = os.path.join(directorio, archivo)
            feats = extraer_caracteristicas_pe(ruta_completa)
            if feats:
                feats['label'] = valor_label
                registros.append(feats)

    df = pd.DataFrame(registros)
    if len(df) > 0:
        print(f"Dataset: {len(df)} muestras | "
              f"Benign={df['label'].eq(0).sum()} | "
              f"Malicious={df['label'].eq(1).sum()}")
    else:
        print("[INFO] No se encontraron archivos PE válidos en los directorios.")
    return df

print("[OK] Función construir_dataset definida.")
print("[INFO] Para usar con archivos reales:")
print("       directorios = {'benign': 'ruta/benign', 'malicious': 'ruta/malicious'}")
print("       df = construir_dataset(directorios)")
print("       df.to_csv('data/file_features.csv', index=False)")

### Dataset sintético: Simulación de características PE

En ausencia de archivos PE reales, se simula un dataset con patrones típicos
de ejecutables benignos y maliciosos, incluyendo secciones, importaciones,
entropía y otros metadatos.

> **En producción**: reemplazar por archivos reales y usar `pefile.PE()` 
> con la función `construir_dataset()` definida previamente.


In [ ]:
import numpy as np
import pandas as pd
import os

np.random.seed(42)
os.makedirs('data', exist_ok=True)

n_benign = 800
n_malicious = 200

# --- Archivos benignos ---
benign = pd.DataFrame({
    'entry_point':        np.random.randint(4096, 65536, n_benign),
    'image_base':         np.full(n_benign, 4194304),               # 0x400000 típico
    'size_of_image':      np.random.randint(50000, 500000, n_benign),
    'size_code_section':  np.random.randint(10000, 200000, n_benign),
    'dll_flag':           np.random.choice([0x8160, 0x8140, 0x8120], n_benign),
    'num_sections':       np.random.choice([3, 4, 5, 6], n_benign, p=[0.1, 0.4, 0.35, 0.15]),
    'entropia_max':       np.random.uniform(4.0, 6.5, n_benign).round(4),
    'entropia_media':     np.random.uniform(3.5, 5.8, n_benign).round(4),
    'num_importaciones':  np.random.randint(50, 400, n_benign),
    'num_dlls_importadas':np.random.randint(5, 25, n_benign),
    'num_exportaciones':  np.random.randint(0, 10, n_benign),
    'file_size':          np.random.randint(50000, 2000000, n_benign),
    'label':              0
})

# --- Archivos maliciosos ---
malicious = pd.DataFrame({
    'entry_point':        np.random.randint(0, 4096, n_malicious),          # entry points bajos / inusuales
    'image_base':         np.random.choice([4194304, 268435456, 65536], n_malicious),
    'size_of_image':      np.random.randint(5000, 100000, n_malicious),
    'size_code_section':  np.random.randint(500, 15000, n_malicious),
    'dll_flag':           np.random.choice([0x0000, 0x0040, 0x8160], n_malicious, p=[0.5, 0.3, 0.2]),
    'num_sections':       np.random.choice([1, 2, 3, 8, 10], n_malicious, p=[0.2, 0.3, 0.2, 0.2, 0.1]),
    'entropia_max':       np.random.uniform(6.5, 8.0, n_malicious).round(4),  # alta entropía = empaquetado
    'entropia_media':     np.random.uniform(5.5, 7.8, n_malicious).round(4),
    'num_importaciones':  np.random.randint(2, 30, n_malicious),              # pocas importaciones
    'num_dlls_importadas':np.random.randint(1, 5, n_malicious),
    'num_exportaciones':  np.random.randint(0, 2, n_malicious),
    'file_size':          np.random.randint(5000, 80000, n_malicious),
    'label':              1
})

# Combinar y mezclar
df_pe = pd.concat([benign, malicious], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
df_pe.to_csv('data/file_features.csv', index=False)

print(f"[OK] Dataset simulado guardado: data/file_features.csv")
print(f"     Total: {len(df_pe)} | Benign: {(df_pe['label']==0).sum()} | Malicious: {(df_pe['label']==1).sum()}")
print(f"\nPrimeras filas:")
df_pe.head()

---

## Entrenamiento y Evaluación de Clasificadores

### Algoritmo 1: Árbol de Decisión (Decision Tree)


In [ ]:
# Listing 4.4: Clasificador de malware con árbol de decisión

import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt

# 1. Cargar datos
df = pd.read_csv('data/file_features.csv').dropna()
X = df.drop('label', axis=1)
y = df['label']

# 2. Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Entrenar
clf = DecisionTreeClassifier(max_depth=10, random_state=42)
clf.fit(X_train, y_train)

# 4. Evaluación
y_pred = clf.predict(X_test)
print("=== Reporte de clasificación ===")
print(classification_report(y_test, y_pred,
                            target_names=['Benign', 'Malicious']))

# Validación cruzada 5-fold
cv_scores = cross_val_score(clf, X, y, cv=5, scoring='f1')
print(f"F1 CV (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# 5. Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Benign', 'Malicious'])
disp.plot(cmap='Blues')
plt.title('Matriz de Confusión - Árbol de Decisión')
plt.tight_layout()
plt.savefig('data/confusion_matrix_dt.png', dpi=150)
plt.show()

In [ ]:
# 6. Importancia de características
importancias = pd.Series(clf.feature_importances_, index=X.columns)
top10 = importancias.nlargest(10)
top10.plot(kind='barh', color='steelblue', figsize=(8, 5))
plt.title('Top 10 características más importantes')
plt.xlabel('Importancia')
plt.tight_layout()
plt.savefig('data/feature_importance.png', dpi=150)
plt.show()

### Algoritmo 2: Random Forest con Balanceo SMOTE


In [ ]:
# Listing 4.5: Random Forest con SMOTE para clases desbalanceadas

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt

# 1. Cargar datos
df = pd.read_csv('data/file_features.csv').dropna()
X, y = df.drop('label', axis=1), df['label']

# 2. Balancear clases con SMOTE (Synthetic Minority Oversampling)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f"Antes SMOTE: {y_train.value_counts().to_dict()}")
print(f"Después SMOTE: {y_train_res.value_counts().to_dict()}")

# 3. Entrenar Random Forest
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train_res, y_train_res)

# 4. Evaluación
y_pred = rf.predict(X_test)
print("\n=== Reporte de clasificación ===")
print(classification_report(y_test, y_pred,
                            target_names=['Benign', 'Malicious']))

# 5. Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Benign', 'Malicious'])
disp.plot(cmap='Oranges')
plt.title('Matriz de Confusión - Random Forest + SMOTE')
plt.tight_layout()
plt.savefig('data/confusion_matrix_rf.png', dpi=150)
plt.show()

---

## Persistencia de Modelos

Se guardan ambos clasificadores y artefactos relacionados para análisis posteriores
de explicabilidad y el pipeline integrado de seguridad.


In [ ]:
import joblib
import os

os.makedirs('models', exist_ok=True)

# Guardar Árbol de Decisión
joblib.dump(clf, 'models/decision_tree_malware.pkl')
print("[OK] Árbol de Decisión guardado en models/decision_tree_malware.pkl")

# Guardar Random Forest
joblib.dump(rf, 'models/random_forest_malware.pkl')
print("[OK] Random Forest guardado en models/random_forest_malware.pkl")

# Guardar nombres de columnas para referencia
joblib.dump(list(X.columns), 'models/malware_feature_names.pkl')
print("[OK] Nombres de características guardados en models/malware_feature_names.pkl")

print("\n=== Resumen de artefactos generados ===")
print("Datos:   data/file_features.csv")
print("Modelos: models/decision_tree_malware.pkl")
print("         models/random_forest_malware.pkl")
print("         models/malware_feature_names.pkl")